In [ ]:
def phen_dataset(mouseMine,KOBAS_identify):
    import pandas as pd
    from tqdm import tqdm
    from collections import defaultdict
    pd.options.mode.chained_assignment = None

    #Creating the list of phenotypes and the final dataframe
    phen = pd.DataFrame(columns=['Phen_ID','Path_ID','Gene_ID'])
    phen.Phen_ID = mouseMine['Allele > Ontology Annotations > Ontology Term . Identifier']
    phen = phen.drop_duplicates(ignore_index=True)

    #Creating the list of genes related to a phenotype
    for i in range(0,len(phen)):
        phen.Gene_ID[i] = mouseMine['Allele > Feature > Primary Identifier'][mouseMine['Allele > Ontology Annotations > Ontology Term . Identifier']==phen.Phen_ID[i]]
        phen.Gene_ID[i] = list(set(phen.Gene_ID[i].values))


    #Creating a dictionary where keys are gene ID and values are list of pathways

    df = KOBAS_identify[['ID','Input_MGI']]
    for i in range(0, len(df)):
        df.loc[i]['Input_MGI'] = (df.loc[i]['Input_MGI'].translate({ord(i): None for i in '[]\'\n'})).split(',')
    d = defaultdict(list)
    for idx, row in df.iterrows():
        gen = row['Input_MGI']
        for g in gen:
            d[g].append(row['ID'])

    for i in tqdm(range(0,len(phen))):
        x=[]
        for key,values in d.items():
            if (key in phen['Gene_ID'][i]):
                x.append(values)
        if not x:
            continue
        else:
            phen.Path_ID[i] = x

    phen.dropna(inplace=True)
    phen.reset_index(inplace=True, drop=True)

    return phen